# PANAV 15N reference distributions by secondary structure

Draws the example panel from the i / i-1 table of makeshift (`makeshift/data/panav_n_prev.csv`).

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import sys
import itertools
MAKESHIFT_ROOT = "../../external/makeshift"
if MAKESHIFT_ROOT not in sys.path:
    sys.path.insert(0, MAKESHIFT_ROOT)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import norm

from makeshift.data.tables import get_panav_n_prev, get_panav_distns

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Arial',
    'mathtext.it': 'Arial:italic',
    'mathtext.bf': 'Arial:bold',
    'svg.fonttype': 'none',   # keep text editable in the exported SVG
})

REF_N = get_panav_n_prev()    # {(aa_i, aa_{i-1}): {ss: (mean, std)}}
REF = get_panav_distns()      # {aa: {ss: {atom: (mean, std)}}}

## Pick a well-separated example

In [ ]:
SS_ORDER = ("H", "E", "C")
MIN_STD = 1.0

rows = []
for (aa, prev), d in REF_N.items():
    if not all(ss in d for ss in SS_ORDER):
        continue
    if min(d[ss][1] for ss in SS_ORDER) < MIN_STD:
        continue
    sep = min(abs(d[a][0] - d[b][0]) / ((d[a][1] + d[b][1]) / 2)
              for a, b in itertools.combinations(SS_ORDER, 2))
    rows.append({"aa": aa, "prev_aa": prev, "min_sep": round(sep, 2),
                 **{f"{ss}_mean": d[ss][0] for ss in SS_ORDER},
                 **{f"{ss}_std": d[ss][1] for ss in SS_ORDER}})

sep_df = pd.DataFrame(rows).sort_values("min_sep", ascending=False).reset_index(drop=True)
sep_df.head(10)

## Plot

In [ ]:
AA, PREV_AA = "LEU", "VAL"
INPUT_N = 121.0


SS_STYLE = {   # label, colour (matches the original panel)
    "H": ("Alpha Helix",  "#F0758A"),
    "E": ("Beta Strand",   "#6DB33F"),
    "C": ("Random Coil",  "#5DADE2"),
}

dist = REF_N[(AA, PREV_AA)]
for ss in SS_ORDER:
    print(f"{SS_STYLE[ss][0]:<12s} mean = {dist[ss][0]:7.2f}  std = {dist[ss][1]:.2f}")

lo = min(m - 4 * s for m, s in dist.values()) + 5
hi = max(m + 4 * s for m, s in dist.values()) + 5
x = np.linspace(lo, hi, 1000)

fig, ax = plt.subplots(figsize=(6, 4))
for ss in SS_ORDER:
    label, color = SS_STYLE[ss]
    mu, sig = dist[ss]
    y = norm.pdf(x, mu, sig)
    ax.plot(x, y, color=color, lw=1.8, label=label)
    ax.fill_between(x, y, color=color, alpha=0.18, lw=0)

if INPUT_N is not None:
    ax.axvline(INPUT_N, color="0.35", ls="--", lw=1.5,
               label=r"Input $\mathregular{^{15}}$N")

ax.set_xlim(lo, hi)
ax.set_ylim(bottom=0)
ax.set_xlabel(r"$\mathregular{^{15}}$N chemical shift (ppm)", fontsize=17)
ax.set_ylabel("Probability density", fontsize=17)
ax.set_yticks([])
ax.set_xticks(np.arange(int(lo), int(hi) + 1, 10))
ax.tick_params(axis='x', labelsize=14)

for side in ("top", "right"):
    ax.spines[side].set_visible(False)
# ax.set_title(f"{AA.title()} (preceded by {PREV_AA.title()})", fontsize=11)
ax.legend(frameon=False, loc="upper right", prop={'family': 'Arial', 'size': 17})

fig.tight_layout()
fig.savefig(f"Figs/panav_N_distn_{AA}_{PREV_AA}.svg", format="svg",
            bbox_inches="tight", transparent=True)
plt.show()

## Comparison: per-residue table (no i-1 dependence)

In [ ]:
d1 = {ss: REF[AA][ss]["N"] for ss in SS_ORDER}
lo1 = min(m - 4 * s for m, s in d1.values()); hi1 = max(m + 4 * s for m, s in d1.values())
x1 = np.linspace(lo1, hi1, 1000)

fig, ax = plt.subplots(figsize=(4.5, 2.6))
for ss in SS_ORDER:
    label, color = SS_STYLE[ss]
    y = norm.pdf(x1, *d1[ss])
    ax.plot(x1, y, color=color, lw=1.8, label=label)
    ax.fill_between(x1, y, color=color, alpha=0.18, lw=0)
ax.set_xlim(lo1, hi1); ax.set_ylim(bottom=0); ax.set_yticks([])
ax.set_xlabel(r"$\mathregular{^{15}}$N chemical shift (ppm)", fontsize=11)
ax.set_ylabel("Probability density", fontsize=11)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.set_title(f"{AA.title()} (per-residue table)", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="upper right")
fig.tight_layout()
plt.show()